In [ ]:
import json
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text
import nltk
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (f1_score, accuracy_score, recall_score,
                             precision_score, confusion_matrix, classification_report)
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from joblib import dump
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.datasets import make_multilabel_classification
from gensim.corpora.dictionary import Dictionary
from gensim.models import LdaModel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
trainset_json = '/content/drive/My Drive/Thesis/Trainset/ML-ESG-3_Trainset_English.json'
testset_json = '/content/drive/My Drive/Thesis/Testset/ML-ESG3_Testset_EN.json'

In [ ]:
with open(trainset_json, 'r') as file:
    train_json_data = json.load(file)

with open(testset_json, 'r') as file:
    test_json_data = json.load(file)

train_df = pd.json_normalize(train_json_data)
test_df = pd.json_normalize(test_json_data)

In [ ]:
train_df.head()

,URL,news_title,news_content,impact_level,impact_length
0,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,ESG-focused financial technology company Arabe...,low,2 to 5 years
1,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,The company also announced the appointment of ...,low,2 to 5 years
2,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,Wong said: \n“Personalised portfolios demand ...,medium,2 to 5 years
3,https://www.esgtoday.com/ukraine-war-inflation...,"Ukraine War, Inflation Reduction Act Driving F...",One of the key themes of the report is the imp...,high,More than 5 years
4,https://www.esgtoday.com/eu-regulators-welcome...,"EU Regulators Welcome, Critique New European S...",Europe’s three primary financial regulatory ag...,medium,Less than 2 years


In [ ]:
train_df["impact_level"] = train_df["impact_level"].replace(["low","medium","high"],[0,1,2])
train_df["impact_length"] = train_df["impact_length"].replace(["Less than 2 years","2 to 5 years","More than 5 years"],[0,1,2])
test_df["impact_level"] = test_df["impact_level"].replace(["low","medium","high"],[0,1,2])
test_df["impact_length"] = test_df["impact_length"].replace(["Less than 2 years","2 to 5 years","More than 5 years"],[0,1,2])

train_df

<ipython-input-6-d9efbd076fbd>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df["impact_level"] = train_df["impact_level"].replace(["low","medium","high"],[0,1,2])
<ipython-input-6-d9efbd076fbd>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df["impact_length"] = train_df["impact_length"].replace(["Less than 2 years","2 to 5 years","More than 5 years"],[0,1,2])
<ipython-input-6-d9efbd076fbd>:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old

,URL,news_title,news_content,impact_level,impact_length
0,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,ESG-focused financial technology company Arabe...,0,1
1,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,The company also announced the appointment of ...,0,1
2,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,Wong said: \n“Personalised portfolios demand ...,1,1
3,https://www.esgtoday.com/ukraine-war-inflation...,"Ukraine War, Inflation Reduction Act Driving F...",One of the key themes of the report is the imp...,2,2
4,https://www.esgtoday.com/eu-regulators-welcome...,"EU Regulators Welcome, Critique New European S...",Europe’s three primary financial regulatory ag...,1,0
...,...,...,...,...,...
540,https://www.esgtoday.com/methane-emissions-det...,Methane Emissions Detection Platform Kuva Rais...,"Stefan Bokaemper, CEO of Kuva Systems, said: “...",0,1
541,https://www.esgtoday.com/eaton-appoints-harold...,Eaton Appoints Harold Jones as Chief Sustainab...,Eaton Appoints Harold Jones as Chief Sustainab...,0,1
542,https://www.esgtoday.com/ssga-outlines-2021-st...,"SSGA Outlines 2021 Stewardship Priorities, Wil...","In his letter, Taraporevala wrote: “As a signa...",1,0
543,https://www.esgtoday.com/survey-investors-shif...,Survey: Investors Shifting to Offense on Clima...,O’Brien said: “Investors globally are increasi...,0,0


In [ ]:
train_data_df = train_df
csv_path = '/content/drive/My Drive/Thesis/Trainset/train_data_df.csv'
train_data_df.to_csv(csv_path, index=False)

test_data_df = test_df
csv_path = '/content/drive/My Drive/Thesis/Testset/test_data_df.csv'
test_data_df.to_csv(csv_path, index=False)

### Latent Dirichlet Allocation (LDA):

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import nltk
from nltk.stem import PorterStemmer

nltk.download('stopwords')
english_stopwords = stopwords.words('english')


nltk.download('omw-1.4')

nltk.download('wordnet')
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    # Lowercase
    text = text.lower()
    tokens = [word for word in text.split() if word not in english_stopwords]
    # Lemmatize tokens
    stem_lemm_tokens = [stemmer.stem(lemmatizer.lemmatize(token)) for token in tokens]

    processed_tokens = [token for token in stem_lemm_tokens if len(token) > 2]
    return processed_tokens


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
train_df["processed_content"] = train_df["news_content"].apply(preprocess_text)
test_df["processed_content"] = test_df["news_content"].apply(preprocess_text)
print(test_df[["news_content", "processed_content"]])

                                          news_content  \
0    Reid Steadman, Managing Director and Global He...   
1    Professor Viner has been appointed as Head of ...   
2    Plastics, chemicals and refining company Lyond...   
3    The company also provided updates towards its ...   
4    Oxender said: ”Designing recyclable coffee pod...   
..                                                 ...   
131  Under the new agreement, Shell will produce re...   
132  Additionally, ENGIE has agreed to aggregate al...   
133  Faerch is a leading European manufacturer of s...   
134  Jean-Yves Duclos, Canada’s Minister of Health,...   
135  Antoine de Saint-Affrique, Chief Executive Off...   

                                     processed_content  
0    [reid, steadman,, manag, director, global, hea...  
1    [professor, viner, appoint, head, gig’, green,...  
2    [plastics,, chemic, refin, compani, lyondellba...  
3    [compani, also, provid, updat, toward, exist, ...  
4    [oxend, said:

In [ ]:
dictionary = Dictionary(train_df["processed_content"])
corpus = [dictionary.doc2bow(text) for text in train_df["processed_content"]]

num_topics = 4
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42)

for i, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"Topic {i + 1}: {topic}")

for i in range(num_topics):
    train_df[f"topic_{i+1}_probability"] = train_df["processed_content"].apply(
        lambda text: dict(lda_model[dictionary.doc2bow(text)]).get(i, 0)
    )

Topic 1: 0.008*"new" + 0.008*"compani" + 0.007*"invest" + 0.006*"includ" + 0.006*"esg" + 0.006*"climat" + 0.006*"global" + 0.005*"said:" + 0.005*"sustain" + 0.004*"food"
Topic 2: 0.008*"energi" + 0.006*"compani" + 0.006*"includ" + 0.005*"new" + 0.005*"invest" + 0.004*"sustain" + 0.004*"financi" + 0.004*"announc" + 0.003*"increas" + 0.003*"also"
Topic 3: 0.010*"sustain" + 0.008*"invest" + 0.008*"includ" + 0.007*"compani" + 0.006*"global" + 0.006*"new" + 0.005*"announc" + 0.005*"manag" + 0.004*"esg" + 0.004*"energi"
Topic 4: 0.010*"compani" + 0.009*"new" + 0.005*"sustain" + 0.005*"water" + 0.005*"includ" + 0.005*"climat" + 0.005*"announc" + 0.004*"esg" + 0.004*"manag" + 0.004*"global"


In [ ]:
for i, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"Topic {i + 1}: {topic}")

Topic 1: 0.008*"new" + 0.008*"compani" + 0.007*"invest" + 0.006*"includ" + 0.006*"esg" + 0.006*"climat" + 0.006*"global" + 0.005*"said:" + 0.005*"sustain" + 0.004*"food"
Topic 2: 0.008*"energi" + 0.006*"compani" + 0.006*"includ" + 0.005*"new" + 0.005*"invest" + 0.004*"sustain" + 0.004*"financi" + 0.004*"announc" + 0.003*"increas" + 0.003*"also"
Topic 3: 0.010*"sustain" + 0.008*"invest" + 0.008*"includ" + 0.007*"compani" + 0.006*"global" + 0.006*"new" + 0.005*"announc" + 0.005*"manag" + 0.004*"esg" + 0.004*"energi"
Topic 4: 0.010*"compani" + 0.009*"new" + 0.005*"sustain" + 0.005*"water" + 0.005*"includ" + 0.005*"climat" + 0.005*"announc" + 0.004*"esg" + 0.004*"manag" + 0.004*"global"


In [ ]:
train_df.head()

,URL,news_title,news_content,impact_level,impact_length,processed_content,topic_1_probability,topic_2_probability,topic_3_probability,topic_4_probability
0,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,ESG-focused financial technology company Arabe...,0,1,"[esg-focus, financi, technolog, compani, arabe...",0.014620,0.956018,0.014704,0.014642
1,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,The company also announced the appointment of ...,0,1,"[compani, also, announc, appoint, tim, wong, n...",0.000000,0.000000,0.000000,0.977752
2,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,Wong said: \n“Personalised portfolios demand ...,1,1,"[wong, said:, “personalis, portfolio, demand, ...",0.126237,0.000000,0.000000,0.863361
3,https://www.esgtoday.com/ukraine-war-inflation...,"Ukraine War, Inflation Reduction Act Driving F...",One of the key themes of the report is the imp...,2,2,"[one, key, theme, report, impact, long-term, o...",0.000000,0.718899,0.270827,0.000000
4,https://www.esgtoday.com/eu-regulators-welcome...,"EU Regulators Welcome, Critique New European S...",Europe’s three primary financial regulatory ag...,1,0,"[europe’, three, primari, financi, regulatori,...",0.000000,0.978508,0.000000,0.000000


testset

In [ ]:
num_topics = 4
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42)

for i, topic in lda_model.print_topics(num_topics=num_topics, num_words=10):
    print(f"Topic {i + 1}: {topic}")

for i in range(num_topics):
    test_df[f"topic_{i+1}_probability"] = test_df["processed_content"].apply(
        lambda text: dict(lda_model[dictionary.doc2bow(text)]).get(i, 0)
    )

Topic 1: 0.009*"compani" + 0.008*"new" + 0.006*"invest" + 0.006*"climat" + 0.006*"includ"
Topic 2: 0.007*"compani" + 0.007*"energi" + 0.006*"includ" + 0.006*"new" + 0.005*"invest"
Topic 3: 0.010*"sustain" + 0.007*"includ" + 0.007*"compani" + 0.007*"invest" + 0.007*"new"


In [ ]:
def topic(row):
    max_prob = max(row["topic_1_probability"], row["topic_2_probability"], row["topic_3_probability"])
    if max_prob > 0.5:
        return row[[f"topic_{i+1}_probability" for i in range(num_topics)]].idxmax().split("_")[1]
    else:
        return 0
train_df["topic"] = train_df.apply(topic, axis=1)
train_df["topic"] = train_df["topic"].astype(int)
test_df["topic"] = test_df.apply(topic, axis=1)
test_df["topic"] = test_df["topic"].astype(int)

In [ ]:
train_df.head()

,URL,news_title,news_content,impact_level,impact_length,processed_content,topic_1_probability,topic_2_probability,topic_3_probability,topic
0,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,ESG-focused financial technology company Arabe...,0,1,"[esg-focus, financi, technolog, compani, arabe...",0.022264,0.954858,0.022892,2
1,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,The company also announced the appointment of ...,0,1,"[compani, also, announc, appoint, tim, wong, n...",0.133567,0.010036,0.884225,3
2,https://www.esgtoday.com/arabesque-ai-appoints...,Arabesque AI Appoints Carolina Minio Paluello ...,Wong said: \n“Personalised portfolios demand ...,1,1,"[wong, said:, “personalis, portfolio, demand, ...",0.831181,0.000000,0.164817,1
3,https://www.esgtoday.com/ukraine-war-inflation...,"Ukraine War, Inflation Reduction Act Driving F...",One of the key themes of the report is the imp...,2,2,"[one, key, theme, report, impact, long-term, o...",0.000000,0.667840,0.320354,2
4,https://www.esgtoday.com/eu-regulators-welcome...,"EU Regulators Welcome, Critique New European S...",Europe’s three primary financial regulatory ag...,1,0,"[europe’, three, primari, financi, regulatori,...",0.010419,0.978217,0.011370,2


In [ ]:
test_df.head()

,URL,news_title,news_content,impact_level,impact_length,processed_content,topic_1_probability,topic_2_probability,topic_3_probability,topic
0,https://www.esgtoday.com/sp-dji-santiago-excha...,"S&P DJI, Santiago Exchange Launch New S&P IPSA...","Reid Steadman, Managing Director and Global He...",1,1,"[reid, steadman,, manag, director, global, hea...",0.201845,0.000000,0.791849,3
1,https://www.esgtoday.com/macquaries-green-inve...,Macquarie’s Green Investment Group Announces 3...,Professor Viner has been appointed as Head of ...,1,1,"[professor, viner, appoint, head, gig’, green,...",0.056996,0.000000,0.938084,3
2,https://www.esgtoday.com/lyondellbasell-launch...,LyondellBasell Launches Suite of Circular Poly...,"Plastics, chemicals and refining company Lyond...",2,1,"[plastics,, chemic, refin, compani, lyondellba...",0.013226,0.011701,0.975091,3
3,https://www.esgtoday.com/orange-commits-to-red...,Orange to Reduce Emissions Across Value Chain ...,The company also provided updates towards its ...,1,1,"[compani, also, provid, updat, toward, exist, ...",0.014751,0.181652,0.803575,3
4,https://www.esgtoday.com/keurig-dr-pepper-achi...,Keurig Dr Pepper Achieves Target of Making 100...,Oxender said: ”Designing recyclable coffee pod...,1,1,"[oxend, said:, ”design, recycl, coffe, pod, on...",0.459936,0.055538,0.482538,0


In [ ]:
train_data_df = train_df
csv_path = '/content/drive/My Drive/Thesis/Trainset/train_topic_df.csv'
train_data_df.to_csv(csv_path, index=False)

test_data_df = test_df
csv_path = '/content/drive/My Drive/Thesis/Testset/test_topic_df.csv'
test_data_df.to_csv(csv_path, index=False)

Let's count how many NONE topic we have.

In [ ]:
zeros = train_df[train_df["topic"] == 0].shape[0]
print(zeros)

4


In [ ]:
X = train_df[['news_content', 'topic']]
y = train_df['impact_level']

levels = train_df.impact_level
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=levels )


In [ ]:
X_train.head()

,news_content,topic
204,ShareAction highlighted several asset managers...,3
357,Why is this important? Even prior to the pande...,2
26,"Additionally, the framework also lists a serie...",3
528,"Andrew Steer, President and CEO, World Resourc...",1
441,It is important to extend the scope of applica...,2


### Approach 1
Taking into account only the Topic column. Let's find the f1-score of the impact level,

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    max_features=2000,
    stop_words=english_stopwords
)

scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', tfidf_vectorizer, 'news_content'),
        ('scaler', scaler, ['topic'])
    ])

svm_params = {
    'C': 1,
    'kernel': 'linear',
    'class_weight': 'balanced',
    'gamma': 'scale'
}

model = make_pipeline(
    preprocessor,
    SVC(**svm_params)
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

print("Classification Report:")
print(classification_report(y_val, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.48      0.56        21
           1       0.57      0.65      0.61        49
           2       0.58      0.56      0.57        39

    accuracy                           0.59       109
   macro avg       0.61      0.56      0.58       109
weighted avg       0.59      0.59      0.59       109



In [ ]:
X_test = test_df[['news_content', 'topic']]
y_test = test_df['impact_level']

model.fit(X_train, y_train)

y_test_pred = model.predict(X_test)

print("Classification Report on Test Set:")
print(classification_report(y_test, y_test_pred))

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.43      0.35      0.39        17
           1       0.57      0.68      0.62        59
           2       0.71      0.62      0.66        60

    accuracy                           0.61       136
   macro avg       0.57      0.55      0.56       136
weighted avg       0.62      0.61      0.61       136



### Aproach 2 :        
Considering the three columns representing the topic probabilities

In [ ]:
X = train_df[['news_content', 'topic_1_probability',	'topic_2_probability',	'topic_3_probability']]
y = train_df['impact_level']

levels = train_df.impact_level
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=levels )

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    max_features=2000,
    stop_words=english_stopwords
)

scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', tfidf_vectorizer, 'news_content'),    #ΔΕΣ ΤΟ standarize ola
        ('scaler', scaler, ['topic_1_probability',	'topic_2_probability',	'topic_3_probability'])
    ])

svm_params = {
    'C': 1,
    'kernel': 'linear',
    'class_weight': 'balanced',
    'gamma': 'scale'
}

model = make_pipeline(
    preprocessor,
    SVC(**svm_params)
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

print("Classification Report:")
print(classification_report(y_val, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.48      0.54        21
           1       0.57      0.63      0.60        49
           2       0.59      0.59      0.59        39

    accuracy                           0.59       109
   macro avg       0.60      0.57      0.58       109
weighted avg       0.59      0.59      0.59       109



In [ ]:
X_test = test_df[['news_content', 'topic_1_probability', 'topic_2_probability', 'topic_3_probability']]
y_test = test_df['impact_level']

model.fit(X_train, y_train)

y_test_pred = model.predict(X_test)

print("Classification Report on Test Set:")
print(classification_report(y_test, y_test_pred))

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.50      0.53      0.51        17
           1       0.57      0.66      0.61        59
           2       0.70      0.58      0.64        60

    accuracy                           0.61       136
   macro avg       0.59      0.59      0.59       136
weighted avg       0.62      0.61      0.61       136



## Multi task

In [ ]:
X_train = train_df['news_content']
y_train_level = train_df['impact_level']
y_train_length = train_df['impact_length']

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np

y_train_combined = np.array(train_df[['impact_level', 'impact_length']])
y_test_combined = np.array(test_df[['impact_level', 'impact_length']])

tfidf_params = {
    'lowercase': True,
    'ngram_range': (1, 1),
    'max_features': 3000,
    'stop_words': english_stopwords
}

base_estimator = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    bootstrap=True
)

multi_output_clf = MultiOutputClassifier(base_estimator)

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(**tfidf_params), 'news_content'),
        ('scaler', scaler, ['topic'])
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('multioutput', multi_output_clf)
])

pipeline.fit(train_df[['news_content', 'topic']], y_train_combined)

y_test_pred_combined = pipeline.predict(test_df[['news_content', 'topic']])

print("Impact Level Classification Report (Test Set):")
print(classification_report(y_test_combined[:, 0], y_test_pred_combined[:, 0]))

print("Impact Length Classification Report (Test Set):")
print(classification_report(y_test_combined[:, 1], y_test_pred_combined[:, 1]))

f1_level = f1_score(y_test_combined[:, 0], y_test_pred_combined[:, 0], average='weighted')
f1_length = f1_score(y_test_combined[:, 1], y_test_pred_combined[:, 1], average='weighted')
print("F1 Score for Impact Level:", f1_level)
print("F1 Score for Impact Length:", f1_length)

f1_combined = (f1_level + f1_length) / 2
print("Combined F1 Score (Average):", f1_combined)

f1_combined_weighted = (f1_level * len(y_test_combined[:, 0]) + f1_length * len(y_test_combined[:, 1])) / (len(y_test_combined[:, 0]) + len(y_test_combined[:, 1]))
print("Combined F1 Score (Weighted):", f1_combined_weighted)

f1_combined_multiply = f1_level * f1_length
print("Combined F1 Score (Multiplication):", f1_combined_multiply)


Impact Level Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.45      0.53      0.49        17
           1       0.51      0.54      0.52        59
           2       0.64      0.57      0.60        60

    accuracy                           0.55       136
   macro avg       0.53      0.55      0.54       136
weighted avg       0.56      0.55      0.55       136

Impact Length Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.19      0.50      0.27         6
           1       0.57      0.55      0.56        47
           2       0.70      0.63      0.66        83

    accuracy                           0.60       136
   macro avg       0.49      0.56      0.50       136
weighted avg       0.63      0.60      0.61       136

F1 Score for Impact Level: 0.5538770928872566
F1 Score for Impact Length: 0.6095354797689851
Combined F1 Score (Average): 0.5817062863281208
Co

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np

y_train_combined = np.array(train_df[['impact_level', 'impact_length']])
y_test_combined = np.array(test_df[['impact_level', 'impact_length']])

tfidf_params = {
    'lowercase': True,
    'ngram_range': (1, 1),
    'max_features': 3000,
    'stop_words': english_stopwords
}

base_estimator = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    bootstrap=True
)

multi_output_clf = MultiOutputClassifier(base_estimator)

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(**tfidf_params), 'news_content'),
        ('scaler', StandardScaler(), ['topic_1_probability', 'topic_2_probability', 'topic_3_probability'])
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('multioutput', multi_output_clf)
])

pipeline.fit(train_df[['news_content', 'topic_1_probability', 'topic_2_probability', 'topic_3_probability']], y_train_combined)

y_test_pred_combined = pipeline.predict(test_df[['news_content', 'topic_1_probability', 'topic_2_probability', 'topic_3_probability']])

print("Impact Level Classification Report (Test Set):")
print(classification_report(y_test_combined[:, 0], y_test_pred_combined[:, 0]))

print("Impact Length Classification Report (Test Set):")
print(classification_report(y_test_combined[:, 1], y_test_pred_combined[:, 1]))

f1_level = f1_score(y_test_combined[:, 0], y_test_pred_combined[:, 0], average='weighted')
f1_length = f1_score(y_test_combined[:, 1], y_test_pred_combined[:, 1], average='weighted')
print("F1 Score for Impact Level:", f1_level)
print("F1 Score for Impact Length:", f1_length)

f1_combined = (f1_level + f1_length) / 2
print("Combined F1 Score (Average):", f1_combined)

f1_combined_weighted = (f1_level * len(y_test_combined[:, 0]) + f1_length * len(y_test_combined[:, 1])) / (len(y_test_combined[:, 0]) + len(y_test_combined[:, 1]))
print("Combined F1 Score (Weighted):", f1_combined_weighted)

f1_combined_multiply = f1_level * f1_length
print("Combined F1 Score (Multiplication):", f1_combined_multiply)


Impact Level Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.35      0.41      0.38        17
           1       0.57      0.68      0.62        59
           2       0.74      0.57      0.64        60

    accuracy                           0.60       136
   macro avg       0.55      0.55      0.55       136
weighted avg       0.62      0.60      0.60       136

Impact Length Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.08      0.17      0.11         6
           1       0.56      0.53      0.54        47
           2       0.70      0.66      0.68        83

    accuracy                           0.60       136
   macro avg       0.45      0.45      0.44       136
weighted avg       0.62      0.60      0.61       136

F1 Score for Impact Level: 0.599354012919044
F1 Score for Impact Length: 0.6071188942565754
Combined F1 Score (Average): 0.6032364535878096
Com

### TF-IDF dataframe

In [ ]:
english_stopwords = "english"
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    max_features=3000,
    stop_words=english_stopwords
)


tfidf_matrix = tfidf_vectorizer.fit_transform(train_df["news_content"])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

In [ ]:
selected_columns = train_df[["topic_1_probability", "topic_2_probability", "topic_3_probability", "impact_level", "impact_length"]]
train_data = pd.concat([selected_columns.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
print(train_data)

     topic_1_probability  topic_2_probability  topic_3_probability  \
0               0.022264             0.954858             0.022892   
1               0.133567             0.010036             0.884225   
2               0.831181             0.000000             0.164817   
3               0.000000             0.667840             0.320354   
4               0.010419             0.978217             0.011370   
..                   ...                  ...                  ...   
540             0.011735             0.011037             0.977240   
541             0.048973             0.489496             0.456267   
542             0.958580             0.000000             0.032853   
543             0.959440             0.020570             0.019951   
544             0.981651             0.000000             0.000000   

     impact_level  impact_length  000  028  0525   10       100  ...  young  \
0               0              1  0.0  0.0   0.0  0.0  0.000000  ...    0.0   
1

In [ ]:
english_stopwords = "english"
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    max_features=3000,
    stop_words=english_stopwords
)


tfidf_matrix = tfidf_vectorizer.fit_transform(test_df["news_content"]) ### fit transform train  kai transform sto test (xwris fit)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

print(tfidf_df)

     000   10       100  109   12  127  130   15  150  180  ...  xpansiv  \
0    0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
1    0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
2    0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
3    0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
4    0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
..   ...  ...       ...  ...  ...  ...  ...  ...  ...  ...  ...      ...   
131  0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
132  0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
133  0.0  0.0  0.126269  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
134  0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   
135  0.0  0.0  0.000000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...      0.0   

        year     years  yie  york  young  youthsource  yum      yves  zero  
0    0.081

In [ ]:
selected_columns = test_df[["topic_1_probability", "topic_2_probability", "topic_3_probability", "impact_level", "impact_length"]]
test_data = pd.concat([selected_columns.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)
print(test_data)

     topic_1_probability  topic_2_probability  topic_3_probability  \
0               0.201845             0.000000             0.791849   
1               0.056996             0.000000             0.938084   
2               0.013226             0.011701             0.975091   
3               0.014751             0.181652             0.803575   
4               0.459936             0.055538             0.482538   
..                   ...                  ...                  ...   
131             0.010412             0.010840             0.978751   
132             0.017219             0.489700             0.493096   
133             0.977610             0.010438             0.011950   
134             0.817166             0.168960             0.015738   
135             0.971416             0.014020             0.014554   

     impact_level  impact_length  000   10       100  109   12  ...  xpansiv  \
0               1              1  0.0  0.0  0.000000  0.0  0.0  ...      0.0   

In [ ]:
y_train_combined = np.array(train_data[['impact_level', 'impact_length']])
y_test_combined = np.array(test_data[['impact_level', 'impact_length']])

base_estimator = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight='balanced',
    bootstrap=True
)

multi_output_clf = MultiOutputClassifier(base_estimator)

pipeline = Pipeline([
    ('multioutput', multi_output_clf)
])

pipeline.fit(train_data[['topic_1_probability', 'topic_2_probability', 'topic_3_probability']], y_train_combined)
y_test_pred_combined = pipeline.predict(test_data[['topic_1_probability', 'topic_2_probability', 'topic_3_probability']])
print("Impact Level Classification Report (Test Set):")
print(classification_report(y_test_combined[:, 0], y_test_pred_combined[:, 0]))

print("Impact Length Classification Report (Test Set):")
print(classification_report(y_test_combined[:, 1], y_test_pred_combined[:, 1]))

# Calculate F1 scores
f1_level = f1_score(y_test_combined[:, 0], y_test_pred_combined[:, 0], average='weighted')
f1_length = f1_score(y_test_combined[:, 1], y_test_pred_combined[:, 1], average='weighted')

print("F1 Score for Impact Level:", f1_level)
print("F1 Score for Impact Length:", f1_length)

f1_combined = (f1_level + f1_length) / 2
print("Combined F1 Score (Average):", f1_combined)

f1_combined_weighted = (f1_level * len(y_test_combined[:, 0]) + f1_length * len(y_test_combined[:, 1])) / (len(y_test_combined[:, 0]) + len(y_test_combined[:, 1]))
print("Combined F1 Score (Weighted):", f1_combined_weighted)

f1_combined_multiply = f1_level * f1_length
print("Combined F1 Score (Multiplication):", f1_combined_multiply)


Impact Level Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.07      0.12      0.09        17
           1       0.40      0.64      0.49        59
           2       0.45      0.08      0.14        60

    accuracy                           0.33       136
   macro avg       0.31      0.28      0.24       136
weighted avg       0.38      0.33      0.29       136

Impact Length Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.07      0.33      0.11         6
           1       0.34      0.57      0.43        47
           2       0.85      0.28      0.42        83

    accuracy                           0.38       136
   macro avg       0.42      0.39      0.32       136
weighted avg       0.64      0.38      0.41       136

F1 Score for Impact Level: 0.2868705577417171
F1 Score for Impact Length: 0.4071989497302142
Combined F1 Score (Average): 0.34703475373596565
C

In [ ]:
train_data_df = train_data
csv_path = '/content/drive/My Drive/Thesis/Trainset/train_tfidf_df.csv'
train_data_df.to_csv(csv_path, index=False)

test_data_df = test_data
csv_path = '/content/drive/My Drive/Thesis/Testset/test_tfidf_df.csv'
test_data_df.to_csv(csv_path, index=False)